#계획

## 서울시 응답소 Web Crawling

https://eungdapso.seoul.go.kr/req/mayor_hope/mayor_hope.do
서울시 응답소 - 시장에게 알린다 탭에 올라와 있는 '공개된 민원' Web Crawling

주소 구성

주소 https://eungdapso.seoul.go.kr/req/mayor_hope/mayor_hope.do?cp=2&sv=&sk=&rceptNo_enc=&startDt=2024-04-09&endDt=2026-04-09

cp : 검색결과 쪽수
startDT, endDT : 검색 시작일-끝일.
- 시작일과 끝일의 길이는 2년을 넘어갈 수 없음 ex) 20240401-20260401 가능. 20240401-20260402 불가능.
- 실제 데이터는 2001년 5월부터 현재까지 있음.



In [ ]:
requests + BeautifulSoup

In [24]:
import requests
from bs4 import BeautifulSoup
import re, time, random

BASE_URL = "https://eungdapso.seoul.go.kr/req/mayor_hope/mayor_hope.do"
DETAIL_URL = "https://eungdapso.seoul.go.kr/req/mayor_hope/mayor_hope_vie.do"

session = requests.Session()
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

# 리스트 페이지 전체 html 가져오기
def get_list_page(cp: int, start_dt: str, end_dt: str):
    params = {"cp": cp, "sv": "", "sk": "", "rceptNo_enc": "",
              "startDt": start_dt, "endDt": end_dt}
    res = session.get(BASE_URL, params=params, headers=headers)
    return res.text

# ── 리스트 페이지 전체 html에서 총 페이지 수 추출
def parse_last_page(html: str) -> int:
    match = re.search(r'var lastCp = (\d+);', html)
    return int(match.group(1)) if match else 1

# ── 현제 리스트 페이지에 보이는 글(rceptNo_enc) 리스트 추출
def parse_rceptNo_list(html: str) -> list[str]:
    # onView('...') 패턴에서 추출
    return re.findall(r"onView\('([^']+)'\)", html)

# 상세 페이지 html 가져오기 : rceptNo_enc = 글 ID
def get_detail_page(rceptNo_enc: str):
    data = {
        "rceptNo_enc": rceptNo_enc,
    }
    res = session.post(DETAIL_URL, data=data, headers=headers) #POST로 rceptNo_enc 전송
    return BeautifulSoup(res.text, "html.parser")

# ── 5. 상세 페이지 html에서 데이터 파싱
def parse_details(soup) -> dict:

    # <br> 태그를 \n으로 바꾸는 헬퍼
    def br_to_newline(tag):
        for br in tag.find_all("br"):
            br.replace_with("\n")
        return tag.get_text()

    # 제목
    title_tag = soup.select_one("h3.sclinkage_title")
    title = title_tag.get_text(strip=True) if title_tag else ""

    # 날짜 (첫 번째 sclinkage_evalue)
    date_tag = soup.select("span.sclinkage_evalue")
    date_val = date_tag[0].get_text(strip=True) if date_tag else ""

    # 상담내용 / 답변내용 — indent_vtit 텍스트로 구분
    question, answer = "", ""
    for block in soup.select("div.indent_vitem"):
        label = block.select_one("p.indent_vtit")
        content = block.select_one("span.rp_indata")
        if not label or not content:
            continue
        text = br_to_newline(content).strip()
        if "상담" in label.get_text():
            question = text
        elif "답변" in label.get_text():
            answer = text

    return {"title": title, "Date": date_val, "Question": question, "Answer": answer}



시험

In [26]:
start_dt = '2001-05-01'
end_dt = '2002-01-01'

#전체 리스트
page_1 = get_list_page(cp = 1, start_dt = start_dt, end_dt = end_dt)
last_page = parse_last_page(page_1)
rceptNo_list = parse_rceptNo_list(page_1)

#상세 페이지
page_1_1 = get_detail_page(rceptNo_list[0])
parse_details(page_1_1)

{'title': '시장님께',
 'Date': '2002-01-01',
 'Question': '서울특별시 시장님 안녕하십니까\n 저는 구미시에 사는 이○○이라고합니다\n 건의할께있어서 이렇게 글을 올립니다\n 도서상품권에 관한것입니다\n 도서상품권은 돈을 주고 사는 것인데\n 왜 도서상품권은 20프로밖에 환불인안돼조\n 그러면 차라리 돈주고 책을 사는게났겟습니다\n 도서상품권은 선물용 이라고도그러시는대\n 누가 쓰던지 마찬가지 아닙니까\n 도서상품권\n 아무런필요가 없다고 생각합니다',
 'Answer': '안녕하십니까? 서울시장 고 건 입니다.\n ○○○님께서는 도서상품권의 잔여환급에 대해 문의하셨습니다.\n 현재 상품권 관련업은 자유업으로 현행법상 제재수단은 없으며 상품권 잔액환급금액에 대한 기준은 "소비자피해보상규정"에 의해 금액상품권의 잔액환급비율 금액이상에 상당하는 물품 또는 용역을 제공받고 그 잔액을 환급할 수 있도록 하고 있습니다.\n 잔액환급비율은 상품권의 권면금액이 1만원 초과일 경우는 100분의 60, 상품권의 권면금액이 1만원이하일 경우는 100분의 80입니다.\n ○○○님의 궁금한 사항에 대해 도움이 되었기를 바라며, 기타 소비자피해와 관련하여 궁금하신 사항이 있으시면 서울시 소비자정보센터(733-9898, 739-9898)로 연락주시면 친절히 도와드리겠습니다.\n 추운 날씨에 건강조심하시기 바랍니다.\n \n 2002. 1. 7\n 서울시장 고 건 드림'}

검색 기간 설정

In [58]:
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

#데이터가 존재하는 2001-05-01부터 현재까지
#검색 기간 설정 제약조건인 2년에 맞도록 기간 구성.
def generate_date_ranges(start: date, end: date):
    # 분기 시작월: 1, 4, 7, 10
    quarter_starts = [1, 4, 7, 10]

    ranges = []
    cur = end

    while cur >= start:
        # 현재 날짜가 속한 분기 시작일 계산
        q_month = ((cur.month - 1) // 3) * 3 + 1
        q_start = date(cur.year, q_month, 1)
        q_start = max(q_start, start)  # start보다 앞으로 가지 않도록

        ranges.append((q_start.strftime("%Y-%m-%d"), cur.strftime("%Y-%m-%d")))

        # 다음 반복: 해당 분기 시작일 하루 전
        cur = q_start - timedelta(days=1)

    return ranges

generate_date_ranges(date(2001,4,1), date(2026,4,9))

[('2026-04-01', '2026-04-09'),
 ('2026-01-01', '2026-03-31'),
 ('2025-10-01', '2025-12-31'),
 ('2025-07-01', '2025-09-30'),
 ('2025-04-01', '2025-06-30'),
 ('2025-01-01', '2025-03-31'),
 ('2024-10-01', '2024-12-31'),
 ('2024-07-01', '2024-09-30'),
 ('2024-04-01', '2024-06-30'),
 ('2024-01-01', '2024-03-31'),
 ('2023-10-01', '2023-12-31'),
 ('2023-07-01', '2023-09-30'),
 ('2023-04-01', '2023-06-30'),
 ('2023-01-01', '2023-03-31'),
 ('2022-10-01', '2022-12-31'),
 ('2022-07-01', '2022-09-30'),
 ('2022-04-01', '2022-06-30'),
 ('2022-01-01', '2022-03-31'),
 ('2021-10-01', '2021-12-31'),
 ('2021-07-01', '2021-09-30'),
 ('2021-04-01', '2021-06-30'),
 ('2021-01-01', '2021-03-31'),
 ('2020-10-01', '2020-12-31'),
 ('2020-07-01', '2020-09-30'),
 ('2020-04-01', '2020-06-30'),
 ('2020-01-01', '2020-03-31'),
 ('2019-10-01', '2019-12-31'),
 ('2019-07-01', '2019-09-30'),
 ('2019-04-01', '2019-06-30'),
 ('2019-01-01', '2019-03-31'),
 ('2018-10-01', '2018-12-31'),
 ('2018-07-01', '2018-09-30'),
 ('2018-

크롤링 main 파이프라인 구성, 실행

In [59]:
import json

def crawl(start_dt, end_dt):
  results = []

  html = get_list_page(cp=1, start_dt=start_dt, end_dt=end_dt)
  last_page = parse_last_page(html)
  print(f"[{start_dt} ~ {end_dt}] 총 {last_page}페이지")
  time.sleep(random.uniform(2.0, 3.5))

  for cp in range(1, last_page + 1):
    page_html = get_list_page(cp, start_dt, end_dt)
    time.sleep(random.uniform(0.8, 1.5))

    for rceptNo_enc in parse_rceptNo_list(page_html):
      detail_html = get_detail_page(rceptNo_enc)
      data = parse_details(detail_html)
      data["rceptNo_enc"] = rceptNo_enc

      results.append(data)
      time.sleep(random.uniform(1.0, 2.0))

    print(f"  {cp}/{last_page} 페이지 완료")

  filename = f"mayor_hope_{start_dt}_{end_dt}.json"
  with open(filename, "w", encoding="utf-8") as f:
      json.dump(results, f, ensure_ascii=False, indent=2)
  print(f"  → {filename} 저장 완료 ({len(results)}건)\n")

In [ ]:
for start_dt, end_dt in generate_date_ranges(date(2001,4,1), date(2026,4,9)):
  crawl(start_dt, end_dt)

In [60]:
for start_dt, end_dt in generate_date_ranges(date(2001,4,1), date(2001,5,31)):
  crawl(start_dt, end_dt)

[2001-04-01 ~ 2001-05-31] 총 1페이지
  1/1 페이지 완료
  → mayor_hope_2001-04-01_2001-05-31.json 저장 완료 (4건)

